In [2]:
from pyflink.datastream import StreamExecutionEnvironment, KeyedProcessFunction, RuntimeContext
from pyflink.table import StreamTableEnvironment, EnvironmentSettings
from pyflink.common import Types, Configuration, Encoder, Row
from pyflink.datastream.state import ValueStateDescriptor
from pyflink.datastream.connectors import FileSink, OutputFileConfig

In [3]:
env = StreamExecutionEnvironment.get_execution_environment()
env.set_parallelism(1)
settings = EnvironmentSettings.new_instance().in_streaming_mode().build()
t_env = StreamTableEnvironment.create(env, environment_settings=settings)

/usr/local/lib/python3.11/dist-packages/apache_beam/runners/portability/stager.py:63: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [4]:
t_env.execute_sql("DROP TABLE IF EXISTS klines_source")
t_env.execute_sql("""
CREATE TABLE klines_source (
    window_start TIMESTAMP(3),
    window_end TIMESTAMP(3),
    open_price DOUBLE,
    high_price DOUBLE,
    low_price DOUBLE,
    close_price DOUBLE,
    volume DOUBLE
) WITH (
    'connector' = 'filesystem',
    'path' = '/workspace/output/klines/ADAUSDT/2025-09-27',
    'format' = 'csv'
)
""")

In [23]:
# klines_source = t_env.from_path("klines_source")
# klines_source.execute().print()
# klines_source.to_pandas().head(20)

# printed = {"done": False}
# def inspect_types(r):
#     if not printed["done"]:
#         print({name: type(value).__name__ for name, value in zip(r._fields, r)})
#         printed["done"] = True
#     return r
# klines_stream = t_env.to_data_stream(t_env.from_path("klines_source"))
# klines_stream.map(inspect_types)
# klines_stream.get_type()
# env.execute("Print Klines Source")

In [4]:
def round_half_up(x, decimals=2):
    if x is None:
        return None
    factor = 10**decimals
    return float(int(x * factor + 0.5)) / factor

class EMA7Function(KeyedProcessFunction):
    def open(self, runtime_context: RuntimeContext):
        self.ema_state = runtime_context.get_state(
            ValueStateDescriptor("ema7", Types.DOUBLE())
        )
        self.buffer = []
        self.alpha = 2 / (7 + 1)

    def process_element(self, value, ctx):
        prev_ema = self.ema_state.value()

        if prev_ema is None:
            self.buffer.append(value["close_price"])
            if len(self.buffer) < 7:
                yield Row(**value.as_dict(), ema7=None) 
                return
            ema = sum(self.buffer) / len(self.buffer)
            self.buffer.clear()
        else:
            ema = self.alpha * value["close_price"] + (1 - self.alpha) * prev_ema

        self.ema_state.update(ema)
        yield Row(**value.as_dict(), ema7=round_half_up(ema, 4)) 

In [5]:
klines_stream = t_env.to_data_stream(t_env.from_path("klines_source")).map(lambda r: Row(**r.as_dict(), symbol="ADAUSDT"))
ema7_typeinfo = Types.ROW_NAMED(
    [
        'window_start',
        'window_end',
        'open_price',
        'high_price',
        'low_price',
        'close_price',
        'volume',
        'ema7',
        'symbol'
    ],
    [
        Types.SQL_TIMESTAMP(),  # TIMESTAMP(3)
        Types.SQL_TIMESTAMP(),
        Types.DOUBLE(),
        Types.DOUBLE(),
        Types.DOUBLE(),
        Types.DOUBLE(),
        Types.DOUBLE(),
        Types.DOUBLE(),
        Types.STRING()
    ]
)
ema_stream = klines_stream.key_by(lambda x: x["symbol"]).process(EMA7Function(), output_type=ema7_typeinfo)

In [6]:
t_env.execute_sql("DROP TABLE IF EXISTS ema7_sink")
t_env.execute_sql("""
CREATE TABLE ema7_sink (
    window_start TIMESTAMP(3),
    window_end TIMESTAMP(3),
    open_price DOUBLE,
    high_price DOUBLE,
    low_price DOUBLE,
    close_price DOUBLE,
    volume DOUBLE,
    ema7 DOUBLE,
    symbol STRING
) WITH (
    'connector' = 'filesystem',
    'path' = '/workspace/output/ema7',
    'format' = 'csv',
     'csv.null-literal' = ''
)
""")

t_env.create_temporary_view("ema_stream", t_env.from_data_stream(ema_stream))
t_env.execute_sql("INSERT INTO ema7_sink SELECT * FROM ema_stream")

In [7]:
t_env.from_path("ema7_sink").to_pandas().head(20)

,window_start,window_end,open_price,high_price,low_price,close_price,volume,ema7,symbol
0,2025-09-27 00:00:00,2025-09-27 00:15:00,0.7918,0.7918,0.7897,0.7901,465779.1,NaN,ADAUSDT
1,2025-09-27 00:15:00,2025-09-27 00:30:00,0.7902,0.7924,0.7900,0.7911,163062.6,NaN,ADAUSDT
2,2025-09-27 00:30:00,2025-09-27 00:45:00,0.7912,0.7920,0.7905,0.7913,202817.4,NaN,ADAUSDT
3,2025-09-27 00:45:00,2025-09-27 01:00:00,0.7913,0.7927,0.7899,0.7923,434212.2,NaN,ADAUSDT
4,2025-09-27 01:00:00,2025-09-27 01:15:00,0.7924,0.7927,0.7902,0.7913,555784.6,NaN,ADAUSDT
5,2025-09-27 01:15:00,2025-09-27 01:30:00,0.7912,0.7922,0.7882,0.7897,550528.6,NaN,ADAUSDT
6,2025-09-27 01:30:00,2025-09-27 01:45:00,0.7898,0.7905,0.7886,0.7899,405461.3,0.7908,ADAUSDT
7,2025-09-27 01:45:00,2025-09-27 02:00:00,0.7900,0.7907,0.7898,0.7903,127489.5,0.7907,ADAUSDT
8,2025-09-27 02:00:00,2025-09-27 02:15:00,0.7903,0.7910,0.7890,0.7909,387130.3,0.7907,ADAUSDT
9,2025-09-27 02:15:00,2025-09-27 02:30:00,0.7909,0.7912,0.7885,0.7896,273607.2,0.7905,ADAUSDT


In [1]:
!jupyter nbconvert --to script test_transform_job_pattern_two.ipynb

[NbConvertApp] Converting notebook test_transform_job_pattern_two.ipynb to script
[NbConvertApp] Writing 4113 bytes to test_transform_job_pattern_two.py
